# 08 · Calibration, stability, and business translation

One unit per dataset × seed (`graph_builder=all`). Reads every saved test/validation prediction for this dataset and
seed from stages 02, 04, 05, 06, 07 and answers the three questions a bank asks after "is the AUC higher?":

1. **Calibration.** Isotonic (and Platt) calibration fitted on validation, applied to test. Brier and ECE before/after,
   reliability curves. A model that ranks better but cannot be calibrated is not deployable.
2. **Stability.** Lending Club has a real time axis: AUC per test quarter and PSI of the score distribution from the
   validation period (2016) to the test period (2017–18). For random-split datasets PSI val→test is a sanity check
   (should be ≈ 0). Subgroup AUC by sex / age where those columns exist (Taiwan, Home Credit).
3. **Business.** At the approval rate where Tier 0 XGBoost approves 80%, what bad rate does it get; at that bad rate,
   how many more applicants can each challenger approve? Plus the swap set: who changes decision, and how did they behave.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 08_calibration_stability_business dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/08_calibration_stability_business/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "taiwan"
seed = 0
graph_builder = "all"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results
from IPython.display import display

STAGE = "08_calibration_stability_business"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Gather predictions

In [ ]:
SOURCE_STAGES = ["02_baselines_tabular", "04_graph_features_xgb", "05_gnn", "06_hybrid", "07_lupi_distill"]
preds = {}   # label -> {"val": df, "test": df}
for stage in SOURCE_STAGES:
    for d in sorted((paths.RUNS_ROOT / stage / dataset).glob(f"*/{seed}")):
        if not ckpt.is_done(d):
            continue
        gb = d.parent.name
        for m in results.list_prediction_models(d):
            if stage == "07_lupi_distill" and m == "xgb_plain":
                continue
            label = f"{stage[:2]}:{gb}:{m}" if gb != "none" else f"{stage[:2]}:{m}"
            preds[label] = {"val": results.load_predictions(d, m, "val"), "test": results.load_predictions(d, m, "test")}
BASE = "02:xgb"
assert BASE in preds, "Tier 0 XGBoost predictions missing; run 02_baselines_tabular first"
print(len(preds), "prediction sets")
pd.Series({k: round(float(__import__("src.eval.metrics", fromlist=["auc"]).auc(v["test"].y, v["test"].p)), 4) for k, v in preds.items()}).sort_values(ascending=False).to_frame("test_auc")

## Calibration

In [ ]:
from src.eval import metrics
from src.eval.calibration import Calibrator, reliability_table

cal_rows, cal_test = {}, {}
for label, d in preds.items():
    yv, pv, yt, pt = d["val"].y.to_numpy(), d["val"].p.to_numpy(), d["test"].y.to_numpy(), d["test"].p.to_numpy()
    iso = Calibrator("isotonic").fit(pv, yv).transform(pt)
    platt = Calibrator("platt").fit(pv, yv).transform(pt)
    cal_test[label] = iso
    cal_rows[label] = {"auc": metrics.auc(yt, pt), "brier_raw": metrics.brier(yt, pt), "brier_isotonic": metrics.brier(yt, iso),
                       "brier_platt": metrics.brier(yt, platt), "ece_raw": metrics.ece(yt, pt), "ece_isotonic": metrics.ece(yt, iso),
                       "ece_platt": metrics.ece(yt, platt), "mean_pd_isotonic": float(iso.mean()), "base_rate": float(yt.mean())}
cal_df = pd.DataFrame(cal_rows).T.sort_values("auc", ascending=False)
show = [BASE] + [k for k in cal_df.index[:4] if k != BASE]
yt = preds[BASE]["test"].y.to_numpy()
fig = viz.plot_reliability({k: reliability_table(preds[k]["test"].y, preds[k]["test"].p) for k in show}, title="raw (test)")
viz.savefig(fig, run_dir, "reliability_raw"); plt.show()
fig = viz.plot_reliability({k: reliability_table(preds[k]["test"].y, cal_test[k]) for k in show}, title="after isotonic (fit on val)")
viz.savefig(fig, run_dir, "reliability_isotonic"); plt.show()
cal_df.round(4)

## Stability

In [ ]:
from src.eval import stability as ST

stab = {"psi_val_to_test": {}, "cohort": {}, "subgroup": {}}
for label, d in preds.items():
    stab["psi_val_to_test"][label] = ST.psi(d["val"].p.to_numpy(), d["test"].p.to_numpy())
psi_s = pd.Series(stab["psi_val_to_test"]).sort_values()
print("PSI val->test (" + ("time-ordered, real drift" if bundle.time is not None else "random split, expect ~0") + "):")
display(psi_s.round(4).to_frame("psi").assign(label=lambda s: s.psi.map(ST.psi_label)))

if bundle.time is not None:
    tq = ST.quarter_cohort(bundle.time)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for label in show:
        t_df = preds[label]["test"]
        c = ST.cohort_metrics(t_df.y, t_df.p, tq[t_df.position.to_numpy()], metrics=("auc",), min_n=300)
        stab["cohort"][label] = c.to_dict(orient="records")
        ax.plot(c["cohort"].astype(str), c["auc"], marker="o", label=label)
    ax.legend(fontsize=7); ax.set(title="Test AUC by issue quarter", ylabel="AUC"); ax.tick_params(axis="x", rotation=45)
    viz.savefig(fig, run_dir, "cohort_auc"); plt.show()

for name, s in bundle.subgroups.items():
    grp = s.to_numpy()
    sub_rows = {}
    for label in show:
        t_df = preds[label]["test"]
        c = ST.cohort_metrics(t_df.y, t_df.p, grp[t_df.position.to_numpy()], metrics=("auc",), min_n=100)
        sub_rows[label] = c.set_index("cohort")["auc"].to_dict()
    stab["subgroup"][name] = sub_rows
    display(pd.DataFrame(sub_rows).round(4).rename_axis(f"subgroup: {name}"))

## Business translation

In [ ]:
from src.eval import business as BZ

bcfg = cfg["business"]
base_t = preds[BASE]["test"].set_index("position")
cands = {}
for label, d in preds.items():
    if label == BASE:
        continue
    t_df = d["test"].set_index("position").reindex(base_t.index)
    if t_df.p.isna().any():
        continue
    cands[label] = t_df.p.to_numpy()
bt = BZ.matched_bad_rate_table(base_t.y.to_numpy(), base_t.p.to_numpy(), cands, base_approval=bcfg["base_approval"])
bt["capture_10"] = [metrics.capture_at(base_t.y, base_t.p, bcfg["capture_frac"])] + [metrics.capture_at(base_t.y, p, bcfg["capture_frac"]) for p in cands.values()]
print(f"Tier 0 XGBoost at {bcfg['base_approval']:.0%} approval has bad rate {bt.attrs['target_bad_rate']:.4f}. At that bad rate:")
fig = viz.plot_bar(bt.set_index("model")["approval_delta_pp"].drop("baseline"), title="Extra approvals (pp) at the baseline bad rate", ylabel="percentage points")
viz.savefig(fig, run_dir, "approval_delta"); plt.show()
best_label = bt.drop(0).sort_values("approval_at_matched_bad_rate", ascending=False)["model"].iloc[0] if len(bt) > 1 else None
swap = BZ.swap_set(base_t.y.to_numpy(), base_t.p.to_numpy(), cands[best_label], bcfg["base_approval"]) if best_label else {}
print("swap set for best challenger", best_label, ":", json.dumps(swap, indent=1))
bt.round(4)

In [ ]:
models_out = {label: {"test": {**metrics.all_metrics(preds[label]["test"].y, preds[label]["test"].p),
                               "brier_isotonic": cal_rows[label]["brier_isotonic"], "ece_isotonic": cal_rows[label]["ece_isotonic"],
                               "psi_val_to_test": stab["psi_val_to_test"][label]}} for label in preds}
results.save_metrics(run_dir, {"models": models_out, "calibration": cal_rows, "stability": stab,
                               "business": {"table": bt.to_dict(orient="records"), "target_bad_rate": bt.attrs["target_bad_rate"],
                                            "best_challenger": best_label, "swap_set": swap},
                               "params": {"seed": seed, "quick": quick, "n_models": len(preds)}})
summary = {"n_models": len(preds), "best_challenger": best_label, "elapsed_s": round(time.time() - t0, 1)}
ckpt.mark_done(run_dir, summary)
summary